# Syntactic Parsing Deep Dive (SNLP Note 04)

Companion notebook for `04-Syntactic-Processing-Parsing.md` (Lecture 4, *Syntactic Processing in NLP*). Every number in the note comes from a cell here. Needs only NLTK (for `Tree` and a reference chart parser).

| Part | Topic |
|---|---|
| A | The slide's CFG: parse *people fish tanks* and *people fish with rods* |
| B | The slide's PCFG: P(t1), P(t2), P(s) for *people fish tanks with rods* |
| C | Binarization / Chomsky Normal Form |
| D | Probabilistic CKY from scratch, with unaries: the full chart for *fish people fish tanks* |
| E | Inside algorithm: P(sentence) = sum over all trees |
| F | Evaluating constituency parses: labeled precision, recall, F1 |
| G | From phrase structure to dependencies with head rules |
| H | Arc-standard and arc-eager transition parsing (the slide's trace) |
| I | UAS and LAS |
| J | Projectivity checker |
| K | How CKY's cost grows |

In [1]:
from itertools import product
from collections import defaultdict
import nltk
from nltk import CFG, PCFG, Tree

## Part A — The phrase-structure grammar from the slide

The slide's grammar has `NP → e` (an empty NP). NLTK's chart parser does not allow empty rules, and no parse of these sentences needs one, so we leave it out.

In [2]:
cfg = CFG.fromstring('''
S -> NP VP
VP -> V NP | V NP PP
NP -> NP NP | NP PP | N
PP -> P NP
N -> 'people' | 'fish' | 'tanks' | 'rods'
V -> 'people' | 'fish' | 'tanks'
P -> 'with'
''')
parser = nltk.ChartParser(cfg)
for sent in ["people fish tanks", "people fish with rods", "people fish tanks with rods"]:
    trees = list(parser.parse(sent.split()))
    print(f"{sent!r}: {len(trees)} parse(s)")
    for t in trees:
        print("   ", t.pformat(margin=200))

'people fish tanks': 1 parse(s)
    (S (NP (N people)) (VP (V fish) (NP (N tanks))))
'people fish with rods': 0 parse(s)
'people fish tanks with rods': 2 parse(s)
    (S (NP (N people)) (VP (V fish) (NP (N tanks)) (PP (P with) (NP (N rods)))))
    (S (NP (N people)) (VP (V fish) (NP (NP (N tanks)) (PP (P with) (NP (N rods))))))


**Why the slide has `NP → e`.** *people fish with rods* gets 0 parses above: the only VP rule with a PP is `VP → V NP PP`, which needs an object NP between *fish* and *with*. An empty NP fills that gap. We simulate the empty word with a visible placeholder `e`:

In [3]:
cfg_e = CFG.fromstring('''
S -> NP VP
VP -> V NP | V NP PP
NP -> NP NP | NP PP | N | 'e'
PP -> P NP
N -> 'people' | 'fish' | 'tanks' | 'rods'
V -> 'people' | 'fish' | 'tanks'
P -> 'with'
''')
for t in nltk.ChartParser(cfg_e).parse("people fish e with rods".split()):
    print(t.pformat(margin=200))

(S (NP (N people)) (VP (V fish) (NP e) (PP (P with) (NP (N rods)))))
(S (NP (N people)) (VP (V fish) (NP (NP e) (PP (P with) (NP (N rods))))))


## Part B — The PCFG: probability of each tree and of the sentence

P(tree) = product of the probabilities of every rule used. P(sentence) = sum of P(tree) over all its trees.

In [4]:
pcfg_rules = {
    ('S', ('NP', 'VP')): 1.0,
    ('VP', ('V', 'NP')): 0.6, ('VP', ('V', 'NP', 'PP')): 0.4,
    ('NP', ('NP', 'NP')): 0.1, ('NP', ('NP', 'PP')): 0.2, ('NP', ('N',)): 0.7,
    ('PP', ('P', 'NP')): 1.0,
    ('N', ('people',)): 0.5, ('N', ('fish',)): 0.2, ('N', ('tanks',)): 0.2, ('N', ('rods',)): 0.1,
    ('V', ('people',)): 0.1, ('V', ('fish',)): 0.6, ('V', ('tanks',)): 0.3,
    ('P', ('with',)): 1.0,
}
# sanity: rules for each left-hand side sum to 1
tot = defaultdict(float)
for (lhs, _), p in pcfg_rules.items(): tot[lhs] += p
print({k: round(v, 10) for k, v in tot.items()})

def tree_prob(t, rules, show=False):
    '''Multiply the probabilities of all rules used in tree t.'''
    p, used = 1.0, []
    for prod in t.productions():
        key = (str(prod.lhs()), tuple(str(s) if isinstance(s, nltk.Nonterminal) else s for s in prod.rhs()))
        used.append(f"{key[0]}->{' '.join(key[1])}={rules[key]}")
        p *= rules[key]
    if show: print("   ", " x ".join(r.split('=')[1] for r in used))
    return p

sent = "people fish tanks with rods".split()
trees = list(parser.parse(sent))
total = 0
for k, t in enumerate(trees, 1):
    p = tree_prob(t, pcfg_rules, show=True)
    total += p
    kind = "verb attachment (VP -> V NP PP)" if "PP" in [c.label() for c in t[1]] else "noun attachment (NP -> NP PP)"
    print(f"t{k}: P = {p:.8f}   {kind}")
    t.pretty_print()
print(f"P(s) = {total:.8f}")

{'S': 1.0, 'VP': 1.0, 'NP': 1.0, 'PP': 1.0, 'N': 1.0, 'V': 1.0, 'P': 1.0}
    1.0 x 0.7 x 0.5 x 0.4 x 0.6 x 0.7 x 0.2 x 1.0 x 1.0 x 0.7 x 0.1
t1: P = 0.00082320   verb attachment (VP -> V NP PP)
        S                      
   _____|_____                  
  |           VP               
  |      _____|_________        
  |     |     |         PP     
  |     |     |     ____|___    
  NP    |     NP   |        NP 
  |     |     |    |        |   
  N     V     N    P        N  
  |     |     |    |        |   
people fish tanks with     rods

    1.0 x 0.7 x 0.5 x 0.6 x 0.6 x 0.2 x 0.7 x 0.2 x 1.0 x 1.0 x 0.7 x 0.1
t2: P = 0.00024696   noun attachment (NP -> NP PP)
              S                    
   ___________|____                 
  |                VP              
  |      __________|___             
  |     |              NP          
  |     |      ________|____        
  |     |     |             PP     
  |     |     |         ____|___    
  NP    |     NP       |      

## Part C — Binarization (the key step towards Chomsky Normal Form)

CKY needs every rule to have at most two children. `VP → V NP PP` becomes `VP → V @VP_V` and `@VP_V → NP PP`. NLTK's `chomsky_normal_form` does the same thing with its own naming (`VP|<NP-PP>`).

In [5]:
t1 = Tree.fromstring("(ROOT (S (NP (N people)) (VP (V fish) (NP (N tanks)) (PP (P with) (NP (N rods))))))")
t1.pretty_print()
b = t1.copy(deep=True)
b.chomsky_normal_form()
b.pretty_print()
print("same words:", t1.leaves() == b.leaves())
b.un_chomsky_normal_form()
print("detransform gives back the original:", b == t1)

       ROOT                    
        |                       
        S                      
   _____|_____                  
  |           VP               
  |      _____|_________        
  |     |     |         PP     
  |     |     |     ____|___    
  NP    |     NP   |        NP 
  |     |     |    |        |   
  N     V     N    P        N  
  |     |     |    |        |   
people fish tanks with     rods

             ROOT                        
              |                           
              S                          
   ___________|____                       
  |                VP                    
  |      __________|______                
  |     |             VP|<NP-PP>         
  |     |      ___________|_______        
  |     |     |                   PP     
  |     |     |            _______|___    
  NP    |     NP          |           NP 
  |     |     |           |           |   
  N     V     N           P           N  
  |     |     |          

## Part D — Probabilistic CKY with unary rules (the slide's chart)

`score[i][j][A]` = best probability of any A-subtree that covers words i..j−1. Three steps:

1. **Lexical:** fill the diagonal cells from `A → word`.
2. **Unaries:** inside each cell, repeatedly try `A → B` (here NP → N, VP → V, S → VP) until nothing improves.
3. **Binary:** for longer spans, try every split point and every rule `A → B C`, then apply unaries again.

In [6]:
cky_grammar = {
    'binary': {('S', 'NP', 'VP'): 0.9, ('VP', 'V', 'NP'): 0.5, ('VP', 'V', '@VP_V'): 0.3, ('VP', 'V', 'PP'): 0.1,
               ('@VP_V', 'NP', 'PP'): 1.0, ('NP', 'NP', 'NP'): 0.1, ('NP', 'NP', 'PP'): 0.2, ('PP', 'P', 'NP'): 1.0},
    'unary': {('S', 'VP'): 0.1, ('VP', 'V'): 0.1, ('NP', 'N'): 0.7},
    'lex': {('N', 'people'): 0.5, ('N', 'fish'): 0.2, ('N', 'tanks'): 0.2, ('N', 'rods'): 0.1,
            ('V', 'people'): 0.1, ('V', 'fish'): 0.6, ('V', 'tanks'): 0.3, ('P', 'with'): 1.0},
}

def apply_unaries(cell, back, G):
    added = True
    while added:
        added = False
        for (A, B), p in G['unary'].items():
            if B in cell and p * cell[B] > cell.get(A, 0):
                cell[A] = p * cell[B]; back[A] = (B,); added = True

def cky(words, G):
    n = len(words)
    score = [[{} for _ in range(n + 1)] for _ in range(n + 1)]
    back = [[{} for _ in range(n + 1)] for _ in range(n + 1)]
    for i, w in enumerate(words):                       # 1. lexical
        for (A, word), p in G['lex'].items():
            if word == w:
                score[i][i+1][A] = p; back[i][i+1][A] = (w,)
        apply_unaries(score[i][i+1], back[i][i+1], G)  # 2. unaries
    for span in range(2, n + 1):                        # 3. binary, short spans first
        for begin in range(0, n - span + 1):
            end = begin + span
            cell = score[begin][end]
            for split in range(begin + 1, end):
                for (A, B, C), p in G['binary'].items():
                    if B in score[begin][split] and C in score[split][end]:
                        prob = score[begin][split][B] * score[split][end][C] * p
                        if prob > cell.get(A, 0):
                            cell[A] = prob; back[begin][end][A] = (split, B, C)
            apply_unaries(cell, back[begin][end], G)
    return score, back

def build_tree(back, words, i, j, A):
    bp = back[i][j][A]
    if len(bp) == 1 and bp[0] in words and j == i + 1 and bp[0] == words[i] and A in {'N', 'V', 'P'}:
        return Tree(A, [bp[0]])
    if len(bp) == 1:
        return Tree(A, [build_tree(back, words, i, j, bp[0])])
    split, B, C = bp
    return Tree(A, [build_tree(back, words, i, split, B), build_tree(back, words, split, j, C)])

words = "fish people fish tanks".split()
score, back = cky(words, cky_grammar)
for span in range(1, len(words) + 1):
    for i in range(len(words) - span + 1):
        j = i + span
        cell = score[i][j]
        print(f"[{i},{j}] {' '.join(words[i:j]):24s}", ", ".join(f"{A}={p:.10g}" for A, p in sorted(cell.items())))
print("\nBest parse, P =", f"{score[0][4]['S']:.10g}")
best = build_tree(back, words, 0, 4, 'S')
best.pretty_print()

[0,1] fish                     N=0.2, NP=0.14, S=0.006, V=0.6, VP=0.06
[1,2] people                   N=0.5, NP=0.35, S=0.001, V=0.1, VP=0.01
[2,3] fish                     N=0.2, NP=0.14, S=0.006, V=0.6, VP=0.06
[3,4] tanks                    N=0.2, NP=0.14, S=0.003, V=0.3, VP=0.03
[0,2] fish people              NP=0.0049, S=0.0105, VP=0.105
[1,3] people fish              NP=0.0049, S=0.0189, VP=0.007
[2,4] fish tanks               NP=0.00196, S=0.0042, VP=0.042
[0,3] fish people fish         NP=6.86e-05, S=0.000882, VP=0.00147
[1,4] people fish tanks        NP=6.86e-05, S=0.01323, VP=9.8e-05
[0,4] fish people fish tanks   NP=9.604e-07, S=0.00018522, VP=2.058e-05

Best parse, P = 0.00018522
                 S                
       __________|________         
      NP                  VP      
  ____|____           ____|____    
 NP        NP        |         NP 
 |         |         |         |   
 N         N         V         N  
 |         |         |         |   
fish     people

In [7]:
# The slide prints 'V -> tanks 0.1' in the chart, but its grammar says 0.3.
# Which one did the slide actually use? Look at VP -> V in cell [3,4]:
print("VP -> V in [3,4]:", round(score[3][4]['VP'], 6), "= 0.1 x 0.3, matching the slide's 0.03")
G2 = {**cky_grammar, 'lex': {**cky_grammar['lex'], ('V', 'tanks'): 0.1}}
s2, _ = cky(words, G2)
print("with V -> tanks = 0.1 the cell would be", round(s2[3][4]['VP'], 6), "-> so 0.1 is a typo for 0.3")

VP -> V in [3,4]: 0.03 = 0.1 x 0.3, matching the slide's 0.03
with V -> tanks = 0.1 the cell would be 0.01 -> so 0.1 is a typo for 0.3


## Part E — Inside algorithm: the same chart with *sum* instead of *max*

Replacing `max` by `+` gives the total probability of the sentence over all of its trees (the unary rules here form no cycle, so applying them once in the order NP → N, VP → V, S → VP is exact).

In [8]:
def inside(words, G):
    n = len(words)
    beta = [[defaultdict(float) for _ in range(n + 1)] for _ in range(n + 1)]
    order = [('NP', 'N'), ('VP', 'V'), ('S', 'VP')]
    def unary(cell):
        for A, B in order:
            cell[A] += G['unary'][(A, B)] * cell[B]
    for i, w in enumerate(words):
        for (A, word), p in G['lex'].items():
            if word == w: beta[i][i+1][A] += p
        unary(beta[i][i+1])
    for span in range(2, n + 1):
        for i in range(n - span + 1):
            j = i + span
            for k in range(i + 1, j):
                for (A, B, C), p in G['binary'].items():
                    beta[i][j][A] += p * beta[i][k][B] * beta[k][j][C]
            unary(beta[i][j])
    return beta

beta = inside(words, cky_grammar)
print(f"P('fish people fish tanks') = {beta[0][4]['S']:.10g}")
print(f"best tree alone            = {score[0][4]['S']:.10g}  ({score[0][4]['S']/beta[0][4]['S']:.1%} of the total)")

P('fish people fish tanks') = 0.0002053884
best tree alone            = 0.00018522  (90.2% of the total)


## Part F — Evaluating a constituency parse (PARSEVAL)

Each tree becomes a set of labeled brackets `(label, start, end)`; preterminals (POS tags) and ROOT are left out. Precision = correct / brackets in the candidate, Recall = correct / brackets in the gold tree.

In [9]:
gold = Tree.fromstring('''(ROOT (S (NP (NNS Sales) (NNS executives)) (VP (VBD were) (VP (VBG examining)
        (NP (DT the) (NNS figures)) (PP (IN with) (NP (JJ great) (NN care))))) (NP (NN yesterday)) (. .)))''')
cand = Tree.fromstring('''(ROOT (S (NP (NNS Sales) (NNS executives)) (VP (VBD were) (VP (VBG examining)
        (NP (DT the) (NNS figures)) (PP (IN with) (NP (JJ great) (NN care) (NN yesterday))))) (. .)))''')

def brackets(t):
    out = []
    def walk(node, start):
        if isinstance(node, str): return start + 1
        end = start
        for ch in node: end = walk(ch, end)
        if not (len(node) == 1 and isinstance(node[0], str)) and node.label() != 'ROOT':
            out.append((node.label(), start, end))
        return end
    walk(t, 0)
    return out

G, C = brackets(gold), brackets(cand)
print("gold     :", G)
print("candidate:", C)
correct = set(G) & set(C)
P, R = len(correct) / len(C), len(correct) / len(G)
F = 2 * P * R / (P + R)
print(f"correct = {sorted(correct)}")
print(f"LP = {len(correct)}/{len(C)} = {P:.1%}   LR = {len(correct)}/{len(G)} = {R:.1%}   F1 = {F:.1%}")
tags = sum(a[1] == b[1] for a, b in zip(gold.pos(), cand.pos()))
print(f"tagging accuracy = {tags}/{len(gold.pos())} = {tags/len(gold.pos()):.1%}")
# unlabeled version: ignore the labels
uG, uC = {(s, e) for _, s, e in G}, {(s, e) for _, s, e in C}
print(f"unlabeled: P = {len(uG & uC)}/{len(uC)}, R = {len(uG & uC)}/{len(uG)}")

gold     : [('NP', 0, 2), ('NP', 4, 6), ('NP', 7, 9), ('PP', 6, 9), ('VP', 3, 9), ('VP', 2, 9), ('NP', 9, 10), ('S', 0, 11)]
candidate: [('NP', 0, 2), ('NP', 4, 6), ('NP', 7, 10), ('PP', 6, 10), ('VP', 3, 10), ('VP', 2, 10), ('S', 0, 11)]
correct = [('NP', 0, 2), ('NP', 4, 6), ('S', 0, 11)]
LP = 3/7 = 42.9%   LR = 3/8 = 37.5%   F1 = 40.0%
tagging accuracy = 11/11 = 100.0%
unlabeled: P = 3/7, R = 3/8


## Part G — Head rules: turning a phrase-structure tree into dependencies

Pick a **head child** in every phrase (the verb in a VP, the noun in an NP, the preposition in a PP, the VP in S). The head word of the head child becomes the head word of the phrase; every other child's head word becomes its **dependent**.

In [10]:
HEAD = {'S': ['VP'], 'VP': ['V', 'VBD', 'VB'], 'NP': ['N', 'NN', 'NNS'], 'PP': ['P', 'IN']}

def head_word(t):
    '''Return (head word index, list of (head, dependent) arcs) for tree t whose leaves are (index, word).'''
    if isinstance(t[0], tuple):          # preterminal
        return t[0][0], []
    arcs, heads = [], []
    for ch in t:
        h, a = head_word(ch); arcs += a; heads.append((ch.label(), h))
    hpos = next(k for k, (lab, _) in enumerate(heads) if lab in HEAD[t.label()])
    h = heads[hpos][1]
    arcs += [(h, d) for k, (_, d) in enumerate(heads) if k != hpos]
    return h, arcs

sent = "The boy put the tortoise on the rug".split()
t = Tree.fromstring("(S (NP (D The) (N boy)) (VP (V put) (NP (D the) (N tortoise)) (PP (P on) (NP (D the) (N rug)))))")
for k, pos in enumerate(t.treepositions('leaves')):
    t[pos] = (k, t[pos])
root, arcs = head_word(t)
print("root:", sent[root])
for h, d in sorted(arcs, key=lambda a: a[1]):
    print(f"  {sent[h]:9s} -> {sent[d]}")

root: put
  boy       -> The
  put       -> boy
  tortoise  -> the
  put       -> tortoise
  put       -> on
  rug       -> the
  on        -> rug


## Part H — Transition-based parsing

A **configuration** is (stack, buffer, arcs). An **oracle** that knows the gold tree tells us the right transition at each step; a real parser (MaltParser) replaces the oracle with a trained classifier.

### H.1 Arc-eager on the slide's sentence

In [11]:
def arc_eager_oracle(words, heads, labels):
    '''words[0] = ROOT. heads[k] = gold head of word k. Prints every step like the slide.'''
    stack, buf, A = [0], list(range(1, len(words))), set()
    has_head = set()
    steps = []
    W = lambda xs: "[" + ", ".join(words[x] for x in xs) + "]"
    while buf:
        s, b = stack[-1], buf[0]
        if s != 0 and heads[s] == b and s not in has_head:
            act = f"LA-{labels[s]}"; A.add((b, s)); has_head.add(s); stack.pop()
        elif heads[b] == s:
            act = f"RA-{labels[b]}"; A.add((s, b)); has_head.add(b); stack.append(buf.pop(0))
        elif s in has_head and any(heads[b] == k or heads[k] == b for k in stack[:-1]):
            act = "Reduce"; stack.pop()
        else:
            act = "Shift"; stack.append(buf.pop(0))
        new = [f"{labels[d]}({words[h]}, {words[d]})" for h, d in A if (h, d) not in {x for st in steps for x in st[3]}]
        steps.append((act, W(stack), W(buf[:3]) + (" ..." if len(buf) > 3 else ""), {(h, d) for h, d in A}))
        print(f"{act:10s} {W(stack):38s} {W(buf[:3]) + (' ...' if len(buf) > 3 else ''):30s} {'+ ' + new[0] if new else ''}")
    print(f"\n{len(steps)} transitions for {len(words)-1} words; arcs found: {len(A)}")
    return A

words  = ["ROOT", "Happy", "children", "like", "to", "play", "with", "their", "friends", "."]
heads  = [None,    2,       3,          0,      5,    3,      5,      8,       6,         3]
labels = [None, "amod", "nsubj", "root", "aux", "xcomp", "prep", "poss", "pobj", "punct"]
A = arc_eager_oracle(words, heads, labels)
print("matches the gold tree:", all((heads[d], d) in A for d in range(1, len(words))))

Shift      [ROOT, Happy]                          [children, like, to] ...       
LA-amod    [ROOT]                                 [children, like, to] ...       + amod(children, Happy)
Shift      [ROOT, children]                       [like, to, play] ...           
LA-nsubj   [ROOT]                                 [like, to, play] ...           + nsubj(like, children)
RA-root    [ROOT, like]                           [to, play, with] ...           + root(ROOT, like)
Shift      [ROOT, like, to]                       [play, with, their] ...        
LA-aux     [ROOT, like]                           [play, with, their] ...        + aux(play, to)
RA-xcomp   [ROOT, like, play]                     [with, their, friends] ...     + xcomp(like, play)
RA-prep    [ROOT, like, play, with]               [their, friends, .]            + prep(play, with)
Shift      [ROOT, like, play, with, their]        [friends, .]                   
LA-poss    [ROOT, like, play, with]               [friends, .]  

### H.2 Arc-standard (the *basic* parser on the slide) on *Book me the morning flight*

In [12]:
def arc_standard_oracle(words, heads, labels):
    '''Slide's basic parser: Left-Arc / Right-Arc combine stack top s with buffer front b.
    Right-Arc adds s -> b, removes s from the stack and puts s back at the front of the buffer.'''
    stack, buf, A = [0], list(range(1, len(words))), set()
    W = lambda xs: "[" + ", ".join(words[x] for x in xs) + "]"
    n_steps = 0
    while buf:
        s = stack[-1] if stack else None; b = buf[0]
        kids_done = lambda h: all((h, d) in A for d in range(1, len(words)) if heads[d] == h)
        if s is not None and s != 0 and heads[s] == b:
            act = f"LA-{labels[s]}"; A.add((b, s)); stack.pop()
        elif s is not None and heads[b] == s and kids_done(b):
            act = f"RA-{labels[b]}"; A.add((s, b)); buf[0] = stack.pop()
        else:
            act = "Shift"; stack.append(buf.pop(0))
        n_steps += 1
        print(f"{act:12s} stack={W(stack):32s} buffer={W(buf)}")
        if buf == [0] and not stack: break
    print(f"\n{n_steps} transitions; arcs: " + ", ".join(f"{labels[d]}({words[h]}, {words[d]})" for h, d in sorted(A, key=lambda a: a[1])))

words2  = ["ROOT", "Book", "me", "the", "morning", "flight"]
heads2  = [None, 0, 1, 5, 5, 1]
labels2 = [None, "root", "iobj", "det", "compound", "dobj"]
arc_standard_oracle(words2, heads2, labels2)

Shift        stack=[ROOT, Book]                     buffer=[me, the, morning, flight]
RA-iobj      stack=[ROOT]                           buffer=[Book, the, morning, flight]
Shift        stack=[ROOT, Book]                     buffer=[the, morning, flight]
Shift        stack=[ROOT, Book, the]                buffer=[morning, flight]
Shift        stack=[ROOT, Book, the, morning]       buffer=[flight]
LA-compound  stack=[ROOT, Book, the]                buffer=[flight]
LA-det       stack=[ROOT, Book]                     buffer=[flight]
RA-dobj      stack=[ROOT]                           buffer=[Book]
RA-root      stack=[]                               buffer=[ROOT]

9 transitions; arcs: root(ROOT, Book), iobj(Book, me), det(flight, the), compound(flight, morning), dobj(Book, flight)


## Part I — UAS and LAS (the slide's *She saw the video lecture*)

UAS: fraction of words with the correct head. LAS: correct head **and** correct label.

In [13]:
toks   = ["She", "saw", "the", "video", "lecture"]
gold   = [(2, "nsubj"), (0, "root"), (5, "det"), (5, "nn"), (2, "dobj")]
parsed = [(2, "nsubj"), (0, "root"), (4, "det"), (5, "nsubj"), (2, "ccomp")]
for w, g, p in zip(toks, gold, parsed):
    print(f"{w:8s} gold {g}  parsed {p}  head {'OK' if g[0]==p[0] else 'x'}  label {'OK' if g==p else 'x'}")
uas = sum(g[0] == p[0] for g, p in zip(gold, parsed)); las = sum(g == p for g, p in zip(gold, parsed))
print(f"UAS = {uas}/5 = {uas/5:.0%}   LAS = {las}/5 = {las/5:.0%}")

She      gold (2, 'nsubj')  parsed (2, 'nsubj')  head OK  label OK
saw      gold (0, 'root')  parsed (0, 'root')  head OK  label OK
the      gold (5, 'det')  parsed (4, 'det')  head x  label x
video    gold (5, 'nn')  parsed (5, 'nsubj')  head OK  label x
lecture  gold (2, 'dobj')  parsed (2, 'ccomp')  head OK  label x
UAS = 4/5 = 80%   LAS = 2/5 = 40%


## Part J — Projectivity

A tree is **projective** if no two arcs cross when all arcs are drawn above the sentence. Equivalent test: for every arc h → d, every word strictly between h and d is a descendant of h.

In [14]:
def crossing_pairs(heads):
    arcs = [(min(h, d), max(h, d), h, d) for d, h in enumerate(heads) if h is not None]
    bad = []
    for a in arcs:
        for b in arcs:
            if a[0] < b[0] < a[1] < b[1]:
                bad.append(((a[2], a[3]), (b[2], b[3])))
    return bad

def show_projectivity(words, heads, name):
    bad = crossing_pairs(heads)
    print(f"{name}: {'projective' if not bad else 'NON-projective'}")
    for (h1, d1), (h2, d2) in bad:
        print(f"   {words[h1]} -> {words[d1]}   crosses   {words[h2]} -> {words[d2]}")

w = ["ROOT", "Who", "did", "Bill", "buy", "the", "coffee", "from", "yesterday", "?"]
h = [None,    7,     4,     4,      0,     6,     4,        4,      4,           4]
show_projectivity(w, h, "Who did Bill buy the coffee from yesterday ?")
show_projectivity(words, heads, "Happy children like to play with their friends .")
w3 = ["ROOT", "Discussion", "of", "the", "outstanding", "issues", "was", "completed", "."]
h3 = [None,    7,            1,    5,     5,             2,        7,     0,           7]
show_projectivity(w3, h3, "Discussion of the outstanding issues was completed .")

Who did Bill buy the coffee from yesterday ?: NON-projective
   from -> Who   crosses   buy -> yesterday
   from -> Who   crosses   buy -> ?
   ROOT -> buy   crosses   from -> Who
Happy children like to play with their friends .: projective
Discussion of the outstanding issues was completed .: projective


## Part K — How CKY's work grows with sentence length

The binary step loops over (begin, split, end) triples, i.e. O(n³) of them, each multiplied by the number of binary rules |G|.

In [15]:
for n in [4, 10, 20, 40, 80]:
    triples = sum(1 for b in range(n) for e in range(b + 2, n + 1) for s in range(b + 1, e))
    print(f"n = {n:2d}: (begin, split, end) triples = {triples:>7,}   formula (n^3 - n)/6 = {(n**3 - n)//6:>7,}   "
          f"x {len(cky_grammar['binary'])} binary rules = {triples*len(cky_grammar['binary']):>9,}")

n =  4: (begin, split, end) triples =      10   formula (n^3 - n)/6 =      10   x 8 binary rules =        80
n = 10: (begin, split, end) triples =     165   formula (n^3 - n)/6 =     165   x 8 binary rules =     1,320
n = 20: (begin, split, end) triples =   1,330   formula (n^3 - n)/6 =   1,330   x 8 binary rules =    10,640
n = 40: (begin, split, end) triples =  10,660   formula (n^3 - n)/6 =  10,660   x 8 binary rules =    85,280
n = 80: (begin, split, end) triples =  85,320   formula (n^3 - n)/6 =  85,320   x 8 binary rules =   682,560
